# Formative 2 · Face recognition

Fine-tune [MobileNetV3-Small](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.mobilenet_v3_small.html) to recognize three team members from nine photos.

Run from the project root or `notebooks/`. If needed, install dependencies with `pip install torch torchvision matplotlib`. The first run downloads the pretrained weights.

In [1]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

SEED = 42
EPOCHS = 20
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "images").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the notebook from formative-2 or its notebooks directory.")
image_dir = project_root / "data" / "images"

weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
prepare_image = weights.transforms()
print(f"Training device: {DEVICE}")

Training device: cpu


## 1. Load and crop the photos

Each row gives the filename, member ID, expression, and crop box. The box uses fractions of the image width and height in left, top, right, bottom order.

In [2]:
photo_records = [
    ("20261111_125925.jpg.jpeg", "member_1", "surprised", (0.13, 0.08, 0.86, 0.85)),
    ("20261111_130039.jpg.jpeg", "member_1", "smiling", (0.13, 0.19, 0.85, 0.95)),
    ("Kelvin Aaron-Onuigbo.jpg.jpeg", "member_1", "neutral", (0.15, 0.11, 0.87, 0.88)),
    ("WhatsApp Image 2026-11-11 at 14.47.13(1).jpeg", "member_2", "neutral", (0.09, 0.34, 0.80, 0.70)),
    ("WhatsApp Image 2026-11-11 at 14.47.13(2).jpeg", "member_2", "smiling", (0.13, 0.32, 0.89, 0.71)),
    ("WhatsApp Image 2026-11-11 at 14.47.13.jpeg", "member_2", "surprised", (0.14, 0.29, 0.86, 0.69)),
    ("WhatsApp Image 2026-11-13 at 10.53.26 (1).jpeg", "member_3", "surprised", (0.04, 0.14, 0.87, 0.91)),
    ("WhatsApp Image 2026-11-13 at 10.53.26 (2).jpeg", "member_3", "neutral", (0.05, 0.12, 0.86, 0.89)),
    ("WhatsApp Image 2026-11-13 at 10.53.26.jpeg", "member_3", "smiling", (0.05, 0.15, 0.88, 0.92)),
]

member_ids = sorted({member for _, member, _, _ in photo_records})
expressions = ("neutral", "smiling", "surprised")
member_to_class = {member: index for index, member in enumerate(member_ids)}
member_labels = torch.tensor([member_to_class[member] for _, member, _, _ in photo_records])

def load_face_crop(filename, relative_box):
    with Image.open(image_dir / filename) as original:
        image = ImageOps.exif_transpose(original).convert("RGB")
    left, top, right, bottom = relative_box
    crop = (
        round(left * image.width),
        round(top * image.height),
        round(right * image.width),
        round(bottom * image.height),
    )
    return ImageOps.pad(
        image.crop(crop), (256, 256), Image.Resampling.BILINEAR, color=(128, 128, 128)
    )

face_crops = [load_face_crop(filename, box) for filename, _, _, box in photo_records]
fig, axes = plt.subplots(len(member_ids), len(expressions), figsize=(7, 7))
for face, (_, member, expression, _) in zip(face_crops, photo_records):
    axis = axes[member_to_class[member], expressions.index(expression)]
    axis.imshow(face)
    axis.set_title(f"{member} · {expression}")
    axis.axis("off")
fig.tight_layout()
plt.show()

## 2. Augment the training photos

Create a horizontal flip, a 10° rotation, and a grayscale copy of each crop. Evaluation uses only the original crops of held-out photos.

In [3]:
augmentation_names = ("Original", "Horizontal flip", "10° rotation", "Grayscale")

def augment_face(face):
    return (
        face,
        ImageOps.mirror(face),
        face.rotate(10, resample=Image.Resampling.BILINEAR, fillcolor=(128, 128, 128)),
        ImageOps.grayscale(face).convert("RGB"),
    )

fig, axes = plt.subplots(1, len(augmentation_names), figsize=(9, 3))
for axis, name, preview in zip(axes, augmentation_names, augment_face(face_crops[0])):
    axis.imshow(preview)
    axis.set_title(name)
    axis.axis("off")
fig.tight_layout()
plt.show()

photo_variants = torch.stack([
    torch.stack([prepare_image(variant) for variant in augment_face(face)]) for face in face_crops
])
original_images = photo_variants[:, 0]

## 3. Fine-tune MobileNetV3-Small

Train the last feature block and the classifier. Keep the earlier layers and batch-normalization statistics fixed.

In [4]:
def create_model(pretrained):
    model = mobilenet_v3_small(weights=weights if pretrained else None)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, len(member_ids))
    return model

def train_model(training_photo_indices):
    torch.manual_seed(SEED)
    model = create_model(pretrained=True).to(DEVICE)
    model.features.requires_grad_(False)
    model.features[-1].requires_grad_(True)
    model.train()
    model.features.eval()

    optimizer = torch.optim.AdamW([
        {"params": model.features[-1].parameters(), "lr": 1e-4},
        {"params": model.classifier.parameters(), "lr": 1e-3},
    ], weight_decay=0.01)
    cross_entropy = nn.CrossEntropyLoss()
    training_images = photo_variants[training_photo_indices].flatten(0, 1)
    training_labels = member_labels[training_photo_indices].repeat_interleave(len(augmentation_names))
    training_batches = DataLoader(
        TensorDataset(training_images, training_labels), batch_size=8, shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )

    for _ in range(EPOCHS):
        for images, labels in training_batches:
            optimizer.zero_grad(set_to_none=True)
            loss = cross_entropy(model(images.to(DEVICE)), labels.to(DEVICE))
            loss.backward()
            optimizer.step()

    return model.eval()

## 4. Evaluate on held-out photos

Train three times, holding out one expression across all three members each time. Photos of each member share a capture setting, so the scores only describe this dataset. Testing unknown people and new settings needs separate photos.

In [ ]:
held_out_logits = torch.empty((len(photo_records), len(member_ids)))
photo_expressions = [expression for _, _, expression, _ in photo_records]
for held_out_expression in expressions:
    training_photo_indices = [
        index for index, expression in enumerate(photo_expressions)
        if expression != held_out_expression
    ]
    held_out_photo_indices = [
        index for index, expression in enumerate(photo_expressions)
        if expression == held_out_expression
    ]
    fold_model = train_model(training_photo_indices)

    with torch.inference_mode():
        expression_logits = fold_model(original_images[held_out_photo_indices].to(DEVICE)).cpu()
    held_out_logits[held_out_photo_indices] = expression_logits
    correct = (expression_logits.argmax(dim=1) == member_labels[held_out_photo_indices]).sum().item()
    
    print(f"Held out {held_out_expression}: {correct}/{len(held_out_photo_indices)} correct")

predictions = held_out_logits.argmax(dim=1)
confusion_matrix = torch.bincount(
    len(member_ids) * member_labels + predictions, minlength=len(member_ids) ** 2
).reshape(len(member_ids), len(member_ids))
f1_by_member = 2 * confusion_matrix.diag() / (
    confusion_matrix.sum(dim=0) + confusion_matrix.sum(dim=1)
).clamp_min(1)

accuracy = (predictions == member_labels).float().mean()
macro_f1 = f1_by_member.mean()
evaluation_loss = nn.functional.cross_entropy(held_out_logits, member_labels)

print(f"\nAccuracy: {accuracy:.3f}")
print(f"Macro F1: {macro_f1:.3f}")
print(f"Cross-entropy loss: {evaluation_loss:.3f}")
print(f"Confusion matrix (true rows, predicted columns): {', '.join(member_ids)}")
print(confusion_matrix)

Held out neutral: 3/3 correct
Held out smiling: 3/3 correct
Held out surprised: 3/3 correct

Accuracy: 1.000
Macro F1: 1.000
Cross-entropy loss: 0.001
Confusion matrix (true rows, predicted columns): member_1, member_2, member_3
tensor([[3, 0, 0],
        [0, 3, 0],
        [0, 0, 3]])


## 5. Train on all nine photos and save

Save the final model and one 576-value embedding per original photo. Reload the checkpoint to verify it can make a prediction.

In [ ]:
final_model = train_model(list(range(len(photo_records))))
with torch.inference_mode():
    feature_maps = final_model.features(original_images.to(DEVICE))
    embeddings = final_model.avgpool(feature_maps).flatten(1).cpu()

checkpoint_path = project_root / "models" / "face_mobilenet_v3_small.pt"
checkpoint_path.parent.mkdir(exist_ok=True)
torch.save({
    "architecture": "mobilenet_v3_small",
    "weights": weights.name,
    "members": member_ids,
    "model_state_dict": {name: value.cpu() for name, value in final_model.state_dict().items()},
}, checkpoint_path)

features_path = project_root / "image_features.csv"
with features_path.open("w", newline="", encoding="utf-8") as output:
    writer = csv.writer(output)
    writer.writerow(["filename", "member_id", "expression"] + [
        f"embedding_{i:03d}" for i in range(embeddings.shape[1])
    ])
    
    for (filename, member, expression, _), embedding in zip(photo_records, embeddings):
        writer.writerow([filename, member, expression, *embedding.tolist()])

print(f"Saved {checkpoint_path.relative_to(project_root)}")
print(f"Saved {features_path.relative_to(project_root)} with {len(embeddings)} rows")

Saved models/face_mobilenet_v3_small.pt
Saved image_features.csv with 9 rows


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
reloaded_model = create_model(pretrained=False).eval()
reloaded_model.load_state_dict(checkpoint["model_state_dict"])

with torch.inference_mode():
    logits = reloaded_model(original_images[:1])

predicted_member = checkpoint["members"][logits.argmax(dim=1).item()]
print(f"Checkpoint smoke test on a training photo: predicted {predicted_member}")

Checkpoint smoke test on a training photo: predicted member_1
